<a href="https://colab.research.google.com/github/77MJohn77/Geospatial-Research-Portfolio/blob/main/LULC_Kayonza_2000_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LULC Mapping for Kayonza (2000–2025)

This notebook performs Land Use/Land Cover classification for Kayonza District using Google Earth Engine and Random Forest classification at 5-year intervals.


In [ ]:
# =====================================================
# INSTALL REQUIRED PACKAGES
# =====================================================

!pip install geemap
!pip install earthengine-api

# =====================================================
# IMPORT LIBRARIES
# =====================================================

import ee
import geemap

# =====================================================
# AUTHENTICATE AND INITIALIZE EARTH ENGINE
# =====================================================

ee.Authenticate()
ee.Initialize(project='dotted-music-307621')

# =====================================================
# LOAD AREA OF INTEREST (AOI)
# =====================================================

AOI = ee.FeatureCollection(
    'projects/dotted-music-307621/assets/KAYONZA'
)

# =====================================================
# DISPLAY MAP
# =====================================================

Map = geemap.Map(center=[-1.85, 30.65], zoom=9)
Map.addLayer(AOI, {}, 'Kayonza AOI')

# =====================================================
# CLOUD MASKING FUNCTIONS
# =====================================================

def maskL5(image):
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0).And(
        qa.bitwiseAnd(1 << 4).eq(0)
    )
    return image.updateMask(mask)

def maskL89(image):
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 3).eq(0).And(
        qa.bitwiseAnd(1 << 4).eq(0)
    )
    return image.updateMask(mask)

# =====================================================
# FUNCTION TO CREATE LANDSAT COMPOSITES
# =====================================================

def getComposite(year):

    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)

    if year <= 2011:

        collection = ee.ImageCollection(
            'LANDSAT/LT05/C02/T1_L2'
        ).filterBounds(AOI).filterDate(start, end).map(maskL5)

        image = collection.median()

        image = image.select(
            ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
            ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
        )

    else:

        collection = ee.ImageCollection(
            'LANDSAT/LC08/C02/T1_L2'
        ).filterBounds(AOI).filterDate(start, end).map(maskL89)

        image = collection.median()

        image = image.select(
            ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
            ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
        )

    image = image.multiply(0.0000275).add(-0.2)

    ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
    ndwi = image.normalizedDifference(['Green', 'NIR']).rename('NDWI')
    ndbi = image.normalizedDifference(['SWIR1', 'NIR']).rename('NDBI')

    composite = image.addBands([ndvi, ndwi, ndbi])

    return composite.clip(AOI)

# =====================================================
# CREATE COMPOSITES
# =====================================================

years = [2000, 2005, 2010, 2015, 2020, 2025]

composites = {}

for year in years:
    composites[year] = getComposite(year)

# =====================================================
# VISUALIZATION PARAMETERS
# =====================================================

visParams = {
    'bands': ['Red', 'Green', 'Blue'],
    'min': 0,
    'max': 0.3
}

for year in years:
    Map.addLayer(composites[year], visParams, f'Landsat {year}')

# =====================================================
# TRAINING DATA
# =====================================================

training = ee.FeatureCollection(
    'projects/dotted-music-307621/assets/training_samples'
)

bands = [
    'Blue', 'Green', 'Red', 'NIR',
    'SWIR1', 'SWIR2',
    'NDVI', 'NDWI', 'NDBI'
]

# =====================================================
# CLASSIFICATION FUNCTION
# =====================================================

def classifyImage(image):

    trainingSample = image.select(bands).sampleRegions(
        collection=training,
        properties=['class'],
        scale=30
    )

    classifier = ee.Classifier.smileRandomForest(
        numberOfTrees=200
    ).train(
        features=trainingSample,
        classProperty='class',
        inputProperties=bands
    )

    classified = image.select(bands).classify(classifier)

    return classified

# =====================================================
# CLASSIFY IMAGES
# =====================================================

classified_maps = {}

for year in years:
    classified_maps[year] = classifyImage(composites[year])

# =====================================================
# COLOR PALETTE
# =====================================================

palette = [
    '0000FF',
    '006400',
    'FFFF00',
    'FF0000',
    'D2B48C',
    '00FFFF'
]

for year in years:

    Map.addLayer(
        classified_maps[year],
        {
            'min': 0,
            'max': 5,
            'palette': palette
        },
        f'LULC {year}'
    )

# =====================================================
# EXPORT TO GOOGLE DRIVE
# =====================================================

for year in years:

    task = ee.batch.Export.image.toDrive(
        image=classified_maps[year],
        description=f'LULC_{year}',
        folder='LULC_KAYONZA',
        fileNamePrefix=f'LULC_{year}',
        region=AOI.geometry(),
        scale=30,
        maxPixels=1e13
    )

    task.start()

    print(f'Export started for {year}')

# =====================================================
# DISPLAY MAP
# =====================================================

Map
